# Video utilities

> Save frames as MP4 and compose labelled side-by-side panel videos, per env (stable-worldmodel API).

In [ ]:
#| default_exp plot.video_utils

In [ ]:
#| hide
from nbdev.showdoc import *

`save_panel_videos` writes one video per env with labelled panels side by side (e.g. the
observation, the goal and a model's prediction). A panel entry per env is a sequence of frames
`(T, H, W, C)`, a still `(H, W, C)` repeated for every frame, or, per agent, a sequence
`(T, num_agents, H, W, C)` shown with the agents side by side. Panels may have different
sizes.

In [ ]:
#| export
# Adapted from stable-worldmodel's wrappers and plotting utilities (https://github.com/galilai-group/stable-worldmodel):
#
# MIT License
#
# Copyright (c) 2026 GalilAI-group
#
# Permission is hereby granted, free of charge, to any person obtaining a copy
# of this software and associated documentation files (the "Software"), to deal
# in the Software without restriction, including without limitation the rights
# to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
# copies of the Software, and to permit persons to whom the Software is
# furnished to do so, subject to the following conditions:
#
# The above copyright notice and this permission notice shall be included in all
# copies or substantial portions of the Software.
#
# THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
# IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
# FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
# AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
# LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
# OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
# SOFTWARE.

from __future__ import annotations

from pathlib import Path

import numpy as np

In [ ]:
#| export
def save_video(path: Path, frames: list[np.ndarray], fps: int = 15) -> None:
    "Write `frames` (`(H, W, C)` uint8) as an MP4 at `path` (libx264)."
    if not len(frames):
        return
    import imageio
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    out = imageio.get_writer(str(path), fps=fps, codec='libx264')
    for f in frames:
        out.append_data(f)
    out.close()


def _panel_frames(p) -> np.ndarray:
    "A panel as `(T, H, W, C)` frames (a still is one frame; per-agent views side by side)."
    p = np.asarray(p)
    if p.ndim == 3:
        return p[None]
    if p.ndim == 5:                                       # (T, agents, H, W, C) -> (T, H, agents * W, C)
        return np.concatenate(list(p.transpose(1, 0, 2, 3, 4)), axis=2)
    return p


def save_panel_videos(video_dir, panels, fps: int = 15) -> None:
    """
    Save `env_<i>.mp4` per env with labelled panels side by side. `panels` maps a label to
    per-env data indexable by env index (see above for the shapes of an entry).
    """
    from PIL import Image, ImageDraw, ImageFont

    video_dir = Path(video_dir)
    video_dir.mkdir(parents=True, exist_ok=True)
    labels = list(panels)
    n_envs = len(panels[labels[0]])

    for i in range(n_envs):
        env_panels = [_panel_frames(panels[label][i]) for label in labels]
        sizes = [p.shape[1:3] for p in env_panels]
        h, w0 = max(s[0] for s in sizes), sizes[0][1]
        pad, gap, lh = max(12, w0 // 14), max(10, w0 // 16), max(22, w0 // 9)
        xs = np.cumsum([pad] + [s[1] + gap for s in sizes[:-1]]).tolist()
        cw = (xs[-1] + sizes[-1][1] + pad + 15) // 16 * 16
        ch = (2 * pad + h + lh + 15) // 16 * 16
        try:
            font = ImageFont.truetype('DejaVuSans.ttf', max(12, w0 // 14))
        except OSError:
            font = ImageFont.load_default()
        y_text = pad + h + max(8, lh // 4)
        T = max(len(p) for p in env_panels)
        composed = []
        for t in range(T):
            c = np.full((ch, cw, 3), 250, dtype=np.uint8)
            for x, (ph, pw), p in zip(xs, sizes, env_panels):
                frame = p[min(t, len(p) - 1)]
                c[pad:pad + ph, x:x + pw] = frame if frame.shape[-1] == 3 else np.repeat(frame, 3, axis=-1)
            img = Image.fromarray(c)
            draw = ImageDraw.Draw(img)
            for x, (_, pw), label in zip(xs, sizes, labels):
                b = draw.textbbox((0, 0), label, font=font)
                draw.text((x + pw // 2 - (b[2] - b[0]) // 2, y_text), label, fill=(130, 130, 130), font=font)
            composed.append(np.array(img))
        save_video(video_dir / f'env_{i}.mp4', composed, fps=fps)

### Tests

In [ ]:
import tempfile, imageio
with tempfile.TemporaryDirectory() as tmp:
    save_video(Path(tmp) / 'empty.mp4', [])
    assert not (Path(tmp) / 'empty.mp4').exists()
    frames = [np.full((32, 48, 3), i * 20, np.uint8) for i in range(5)]
    save_video(Path(tmp) / 'a' / 'clip.mp4', frames, fps=5)
    assert len(imageio.mimread(Path(tmp) / 'a' / 'clip.mp4')) == 5

    T, A = 6, 2
    panels = {'views': np.random.randint(0, 255, (3, T, A, 32, 32, 3), np.uint8),    # per agent, side by side
              'render': np.random.randint(0, 255, (3, T, 48, 48, 3), np.uint8),     # another size
              'goal': np.zeros((3, 32, 32, 3), np.uint8)}                           # a still
    save_panel_videos(Path(tmp) / 'panels', panels, fps=5)
    files = sorted(p.name for p in (Path(tmp) / 'panels').iterdir())
    assert files == ['env_0.mp4', 'env_1.mp4', 'env_2.mp4']
    video = imageio.mimread(Path(tmp) / 'panels' / 'env_0.mp4')
    assert len(video) == T and video[0].shape[1] >= 2 * 32 + 48 + 32

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()